<a href="https://colab.research.google.com/github/shanusushmita/CS4973-Applied-Multilingual-Systems/blob/main/Code_Switching_Data_Bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. Define the parameters for the mock dataset
N_SAMPLES = 1000
LANG_DOMINANCE = ['Spanish-Dominant', 'Mandarin-Dominant']
np.random.seed(42) # for reproducibility

# --- INTRODUCING THE BIAS ---
# SCENARIO: The data collection effort was biased.
# For Spanish-Dominant speakers, the collector actively sought out code-switching examples.
# For Mandarin-Dominant speakers, the collector mostly captured monolingual sentences.

data = []
for _ in range(N_SAMPLES):
    # Randomly select a speaker dominance (Sensitive Attribute A)
    primary_lang = np.random.choice(LANG_DOMINANCE)

    # 2. Assign the 'Code_Switching' label (Target Y) based on the introduced bias
    if primary_lang == 'Spanish-Dominant':
        # High probability of code-switching for this group (70%)
        is_code_switching = np.random.choice([1, 0], p=[0.70, 0.30])
    else: # Mandarin-Dominant
        # Low probability of code-switching for this group (30%)
        is_code_switching = np.random.choice([1, 0], p=[0.30, 0.70])

    data.append({
        'Primary_Language': primary_lang,
        'Code_Switching': is_code_switching,
        # Mock feature: Complexity (Higher complexity might correlate slightly with switching)
        'Sentence_Complexity': np.random.normal(5 + is_code_switching*2, 1.5)
    })

df = pd.DataFrame(data)

## 3. Show the resulting Biased Label Distribution
print("## Biased Label Distribution by Primary Language ##")
bias_check = df.groupby('Primary_Language')['Code_Switching'].agg(['mean', 'count'])
bias_check['mean'] = bias_check['mean'].round(3) # The mean is the rate of code-switching (Y=1)
print(bias_check)
print("-" * 50)

# 4. Demonstrate the Predictive Bias
# A simple model will learn to rely on the proxy feature (Primary_Language)

# Encode the Sensitive Attribute (Primary_Language) for the model
df_encoded = pd.get_dummies(df, columns=['Primary_Language'], drop_first=True)

X = df_encoded[['Sentence_Complexity', 'Primary_Language_Spanish-Dominant']]
y = df_encoded['Code_Switching']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train a simple Logistic Regression Model
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# 5. Evaluate Model Performance and Bias (The Harm)
print("##  Model Performance on Test Set ##")
# Note: The model's overall accuracy might look good, but the fairness is poor.
# The coefficient for the primary language will be high, showing it's a proxy.
print(f"Overall Accuracy: {model.score(X_test, y_test):.3f}")
print("-" * 50)

print("## Model Fairness Check (The Bias Harm) ##")
for lang in LANG_DOMINANCE:
    # Filter the test set for the sensitive attribute group
    lang_filter = X_test[f'Primary_Language_Spanish-Dominant'] if lang == 'Spanish-Dominant' else (1 - X_test[f'Primary_Language_Spanish-Dominant'])

    X_lang = X_test[lang_filter == 1]
    y_lang = y_test[lang_filter == 1]
    y_pred_lang = model.predict(X_lang)

    if not y_lang.empty:
        # Calculate the code-switching prediction rate for this group
        pred_rate = y_pred_lang.mean()
        # Calculate the True Positive Rate (Recall)
        # TPR = TP / (TP + FN)
        tpr = np.sum((y_pred_lang == 1) & (y_lang == 1)) / np.sum(y_lang == 1)

        print(f"[{lang} Speakers]")
        print(f"  Predicted Code-Switching Rate: {pred_rate:.3f}")
        print(f"  True Positive Rate (Recall): {tpr:.3f}") # Check Equality of Opportunity

## Biased Label Distribution by Primary Language ##
                    mean  count
Primary_Language               
Mandarin-Dominant  0.282    461
Spanish-Dominant   0.707    539
--------------------------------------------------
##  Model Performance on Test Set ##
Overall Accuracy: 0.797
--------------------------------------------------
## Model Fairness Check (The Bias Harm) ##
[Spanish-Dominant Speakers]
  Predicted Code-Switching Rate: 0.768
  True Positive Rate (Recall): 0.891
[Mandarin-Dominant Speakers]
  Predicted Code-Switching Rate: 0.248
  True Positive Rate (Recall): 0.611
